In [1]:
import os, zipfile, time, gc, random, re
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss

# Remove outdated protobuf hack that raises AttributeError on recent versions
# (the protobuf library already provides the needed functionality)
# If future compatibility is needed, it can be added here safely.

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0

tf.get_logger().setLevel("ERROR")
gpus = tf.config.experimental.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

start = time.time()




2026-05-03 06:07:22.734746: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777788442.748348      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777788442.752443      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-03 06:07:22.769100: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
BASE_PATH = "/kaggle/input/dogs-vs-cats-redux-kernels-edition"

# Prefer already‑extracted folders; fall back to unzipping if they are missing
TRAIN_DIR = os.path.join(BASE_PATH, "train")
TEST_DIR = os.path.join(BASE_PATH, "test")

if not os.path.isdir(TRAIN_DIR) or not os.path.isdir(TEST_DIR):
    # unzip to a temporary location
    extract_dir = "./data"
    os.makedirs(extract_dir, exist_ok=True)

    train_zip = os.path.join(BASE_PATH, "train.zip")
    test_zip = os.path.join(BASE_PATH, "test.zip")

    if os.path.isfile(train_zip):
        with zipfile.ZipFile(train_zip, "r") as z:
            z.extractall(extract_dir)
    if os.path.isfile(test_zip):
        with zipfile.ZipFile(test_zip, "r") as z:
            z.extractall(extract_dir)

    # locate the first "train" and "test" directories inside the extracted tree
    for root, dirs, _ in os.walk(extract_dir):
        if "train" in dirs and not os.path.isdir(TRAIN_DIR):
            TRAIN_DIR = os.path.join(root, "train")
        if "test" in dirs and not os.path.isdir(TEST_DIR):
            TEST_DIR = os.path.join(root, "test")

assert TRAIN_DIR is not None and os.path.isdir(
    TRAIN_DIR
), f"Train directory not found: {TRAIN_DIR}"
assert TEST_DIR is not None and os.path.isdir(
    TEST_DIR
), f"Test directory not found: {TEST_DIR}"

print(f"Train dir: {TRAIN_DIR}")
print(f"Test  dir: {TEST_DIR}")




Train dir: /kaggle/input/dogs-vs-cats-redux-kernels-edition/train
Test  dir: /kaggle/input/dogs-vs-cats-redux-kernels-edition/test


In [3]:
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 32
SEED = 42




In [4]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
    validation_split=0.2,
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training",
    seed=SEED,
    shuffle=True,
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    seed=SEED,
    shuffle=False,
)




Found 18001 images belonging to 3 classes.
Found 4499 images belonging to 3 classes.


In [5]:
base_model = EfficientNetB0(
    weights="imagenet", include_top=False, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)
base_model.trainable = False  # freeze base for faster training

model = models.Sequential(
    [base_model, layers.GlobalAveragePooling2D(), layers.Dense(1, activation="sigmoid")]
)

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()




I0000 00:00:1777788449.564823      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:46:00.0, compute capability: 8.6


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 4, 4, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,050,852 (15.45 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [6]:
early_stop = callbacks.EarlyStopping(
    patience=5, restore_best_weights=True, monitor="val_loss"
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1
)




In [7]:
steps_per_epoch = np.ceil(train_generator.samples / BATCH_SIZE)
validation_steps = np.ceil(val_generator.samples / BATCH_SIZE)

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=20,
    validation_data=val_generator,
    validation_steps=validation_steps,
    callbacks=[early_stop, reduce_lr],
    verbose=2,
)




/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20


TypeError: 'numpy.float64' object cannot be interpreted as an integer

In [8]:
val_preds = model.predict(val_generator, steps=validation_steps, verbose=0).ravel()
val_labels = val_generator.labels[: len(val_preds)]  # align lengths
print("Validation log loss:", log_loss(val_labels, val_preds))




TypeError: 'numpy.float64' object cannot be interpreted as an integer

In [9]:
test_image_paths = []
for root, _, files in os.walk(TEST_DIR):
    for f in files:
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            test_image_paths.append(os.path.join(root, f))

test_image_paths.sort(
    key=lambda x: int(re.search(r"(\d+)", os.path.basename(x)).group(1))
)
print(f"Found {len(test_image_paths)} test images.")

test_array = np.empty(
    (len(test_image_paths), IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.float32
)
for idx, img_path in enumerate(test_image_paths):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT), interpolation=cv2.INTER_CUBIC)
    test_array[idx] = img / 255.0
    if idx % 500 == 0:
        print(f"Loaded {idx}/{len(test_image_paths)}")




Found 2500 test images.
Loaded 0/2500
Loaded 500/2500
Loaded 1000/2500
Loaded 1500/2500
Loaded 2000/2500


In [10]:
test_preds = model.predict(test_array, batch_size=BATCH_SIZE, verbose=1).ravel()

ids = [int(re.search(r"(\d+)", os.path.basename(p)).group(1)) for p in test_image_paths]
submission = pd.DataFrame({"id": ids, "label": test_preds})
submission = submission.sort_values("id")
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)

print(f"Submission file written to {submission_path}")
print(f"Total time: {time.time() - start:.2f} seconds")

I0000 00:00:1777788457.187726     178 service.cc:148] XLA service 0x7f17c0088410 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777788457.187780     178 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-03 06:07:37.258456: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777788457.590425     178 cuda_dnn.cc:529] Loaded cuDNN version 90300


27/79 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

I0000 00:00:1777788460.521302     178 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


79/79 ━━━━━━━━━━━━━━━━━━━━ 11s 71ms/step
Submission file written to submission.csv
Total time: 16.82 seconds
